# Gemma 4 Good - Impact Track
## Climate-Smart Agriculture Assistant

This notebook demonstrates using Gemma 4 27B to build a climate-smart agronomist AI that helps farmers make sustainable farming decisions.

In [ ]:
# Install required packages
!pip install -q transformers accelerate peft bitsandbytes

In [ ]:
import json

import torch
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig


# Load Gemma 4 from Kaggle attached model
model_path = "/kaggle/input/gemma/"

# Use 4-bit quantization for efficient inference
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(
    model_path, quantization_config=bnb_config, device_map="auto", trust_remote_code=True
)

print(f"Loaded Gemma 4 with {sum(p.numel() for p in model.parameters()) / 1e9:.1f}B parameters")

In [ ]:
# Define climate-smart agriculture prompts
TRAINING_PROMPTS = [
    {
        "instruction": "What are the best drought-resistant crops for a farmer in Kenya with limited water access?",
        "response": "Based on your region's semi-arid climate and water constraints, I recommend: 1) Sorghum - requires 30% less water than maize, 2) Cowpeas - nitrogen-fixing and drought-tolerant, 3) Pigeon peas - deep root system accessing groundwater, 4) Finger millet - thrives in low rainfall. Implement drip irrigation and mulch to reduce evaporation by 40%.",
    },
    {
        "instruction": "How can a smallholder farmer reduce post-harvest losses?",
        "response": "Post-harvest loss reduction strategies: 1) Use hermetic storage bags to cut losses by 80%, 2) Dry grains to 13% moisture before storage, 3) Construct raised platforms to prevent pest access, 4) Apply neem oil as natural pesticide, 5) Join farmer cooperatives for collective storage facilities.",
    },
    {
        "instruction": "Recommend sustainable pest management for organic tomato farming",
        "response": "Organic pest management for tomatoes: 1) Companion planting with basil and marigold, 2) Neem-based biopesticides every 14 days, 3) Yellow sticky traps for whiteflies, 4) Introduce ladybugs for aphid control, 5) Crop rotation with legumes to break pest cycles.",
    },
    {
        "instruction": "What soil amendments improve carbon sequestration?",
        "response": "Soil carbon sequestration methods: 1) Add compost - increases soil organic carbon by 0.5-1 ton/ha/year, 2) Cover crops between seasons, 3) No-till farming preserves soil structure, 4) Biochar application - stable carbon for centuries, 5) Agroforestry integration increases root biomass.",
    },
    {
        "instruction": "How to detect plant disease using visual symptoms?",
        "response": "Disease identification: Yellowing leaf edges + mosaic pattern = viral infection. Dark spots with yellow halos = fungal. Wilting despite watering = root rot. White powdery coating = powdery mildew. Remove affected leaves, apply copper-based fungicide, ensure proper spacing for airflow.",
    },
    {
        "instruction": "Calculate water needs for 2-acre mixed vegetable farm",
        "response": "Water calculation: Leafy greens need 25mm/week, tomatoes 35mm/week, root vegetables 20mm/week. For 2 acres mixed: install drip irrigation with 8L/hour emitters, mulch heavily, harvest rainwater. Estimated need: 15,000-20,000L weekly depending on crop mix and evaporation rates.",
    },
    {
        "instruction": "Best practices for regenerative agriculture?",
        "response": "Regenerative practices: 1) Minimize soil disturbance, 2) Maintain soil cover year-round, 3) Maximize plant diversity, 4) Integrate livestock for natural fertilization, 5) Adapt to local ecological context. These rebuild soil organic matter and enhance ecosystem services.",
    },
    {
        "instruction": "How to start a community seed bank?",
        "response": "Community seed bank setup: 1) Collect indigenous varieties from 20+ farmers, 2) Document germination rates and traits, 3) Store in cool dry conditions with silica gel, 4) Establish democratic governance structure, 5) Hold annual seed exchange fairs. Preserves biodiversity and reduces input costs.",
    },
    {
        "instruction": "Climate adaptation for coastal farming?",
        "response": "Coastal adaptation: 1) Salt-tolerant rice varieties (SR-26, Pokkali), 2) Raised bed cultivation for drainage, 3) Mangrove buffer zones, 4) Rainwater harvesting systems, 5) Diversify income with aquaculture integration.",
    },
    {
        "instruction": "Reduce synthetic fertilizer use",
        "response": "Fertilizer reduction: 1) Soil testing before application, 2) Legume intercropping for N-fixation, 3) Vermicomposting on-farm, 4) Green manure crops, 5) Precision application using leaf color charts. Can cut synthetic use by 40-60% while maintaining yield.",
    },
]

# Create training dataset
with open("training_data.json", "w") as f:
    json.dump(TRAINING_PROMPTS, f, indent=2)

print(f"Created {len(TRAINING_PROMPTS)} training examples")

In [ ]:
# Fine-tune with LoRA

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=64,
    lora_alpha=128,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
)

model = get_peft_model(model, lora_config)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# Training setup
from transformers import Trainer, TrainingArguments


training_args = TrainingArguments(
    output_dir="/kaggle/working/gemma4_impact_adapter",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=5,
    save_strategy="epoch",
    remove_unused_columns=False,
)


# Format data for training
def format_example(example):
    text = f"<start_of_turn>user\n{example['instruction']}<end_of_turn>\n<start_of_turn>model\n{example['response']}<end_of_turn>"
    return {"text": text}


import datasets


train_data = datasets.Dataset.from_list(TRAINING_PROMPTS).map(format_example)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    data_collator=lambda x: {
        "input_ids": tokenizer([i["text"] for i in x], return_tensors="pt", padding=True)[
            "input_ids"
        ]
    },
)

trainer.train()

# Save adapter
model.save_pretrained("/kaggle/working/gemma4_impact_adapter")
print("Adapter saved!")

In [ ]:
# Package submission
import shutil


shutil.make_archive("/kaggle/working/submission", "zip", "/kaggle/working/gemma4_impact_adapter")
print("Created submission.zip!")